# Column Tranformer

In ML problems, it is inevitable that data transformation is to be applied to the data and each column might need a different type of transformation. To apply those transformations, the column needs to first be extracted, then transformed then same happens to every column. In this process, all the columns have now been converted to numpy arrays, which now need to be combined till you get a large, single array and then convert that back into a dataframe.

This is too much labor. The ColumnTransformer class of sklearn makes this task easy and quick and allows us to apply different tranformations to each column individually. 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('datasets/covid_toy.csv')
df.sample(5)

,age,gender,fever,cough,city,has_covid
82,24,Male,98.0,Mild,Kolkata,Yes
60,24,Female,102.0,Strong,Bangalore,Yes
70,68,Female,101.0,Strong,Delhi,No
34,74,Male,102.0,Mild,Mumbai,Yes
20,12,Male,98.0,Strong,Bangalore,No


In [4]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

### Applying : 
- simple imputation(filling missing values) to fever
- OHE to gender, city
- Ordinal encoding to cough 

In [10]:
# first train-test split
X = df.iloc[: , :-1]
y = df.iloc[: , -1]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.head())

    age  gender  fever cough       city
45   72    Male   99.0  Mild  Bangalore
80   14  Female   99.0  Mild     Mumbai
54   60  Female   99.0  Mild     Mumbai
3    31  Female   98.0  Mild    Kolkata
85   16  Female  103.0  Mild  Bangalore


In [6]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [19]:
# Applying transformation using the column transformer

# list of columns to apply transformation on, the argument for the transformer object
#Each transformation passed as a tupple in the list
#Format of each tupple : ('name of the transformer (how you want to call it), transformer form (transformation), [list of columns to apply the transformation to])
transformers = [
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(drop='first', sparse_output=False), ['gender', 'city'])
]

#create the column transformer object
transformer = ColumnTransformer(transformers=transformers, remainder='passthrough') # remainder is the attribute about what to do with the cols on which no transformation is being applied -> takes either 'passthrough'or 'drop'

# apply the transformation to the datasets 
X_train_transformed = pd.DataFrame(transformer.fit_transform(X_train))
print(X_train_transformed)

        0    1    2    3    4    5     6
0    99.0  0.0  1.0  0.0  0.0  0.0  72.0
1    99.0  0.0  0.0  0.0  0.0  1.0  14.0
2    99.0  0.0  0.0  0.0  0.0  1.0  60.0
3    98.0  0.0  0.0  0.0  1.0  0.0  31.0
4   103.0  0.0  0.0  0.0  0.0  0.0  16.0
..    ...  ...  ...  ...  ...  ...   ...
75  101.0  0.0  0.0  0.0  0.0  0.0  38.0
76  101.0  0.0  0.0  0.0  0.0  1.0  65.0
77  103.0  0.0  0.0  0.0  1.0  0.0  48.0
78   98.0  1.0  1.0  0.0  0.0  1.0  23.0
79  101.0  0.0  0.0  1.0  0.0  0.0  49.0

[80 rows x 7 columns]
